# Data preprocess

In [49]:
import os
import librosa
import numpy as np
import pandas as pd
import json
from sklearn.model_selection import train_test_split
import xml.etree.ElementTree as ET

## Configurações globais

In [50]:
# Configurações globais
SAMPLE_RATE = 44100
N_FFT = 2048
HOP_LENGTH = 512
SPEC_MAX_LEN = 256  # Número máximo de frames para padronizar os espectrogramas

## Caminhos para as pastas

In [51]:
# Caminho para as pastas principais
DATASET_PATH = "../data/IDMT-SMT-GUITAR_V2/dataset1"
OUTPUT_PATH = "../data/IDMT-SMT-GUITAR_V2/dataset1/preprocessed_data"
LEAVE_OUT_CATEGORY = "Ibanez Power Strat Clean Neck HU"  # Categoria a ser excluída do treino

## Normalização do espectograma

A normalização ajusta os valores de amplitude para uma escala uniforme, permitindo uma comparação mais clara de diferentes sons ou segmentos de áudio. Remove o impacto de variações de volume no treinamento de um modelo.

In [52]:
def normalize_spectrogram(spectrogram):
    """
    Normaliza o espectrograma para o intervalo [0, 1].
    """
    return (spectrogram - spectrogram.min()) / (spectrogram.max() - spectrogram.min())

## Parse do xml

Aqui é só para retirar as informações relevantes da notas presentes nos xml para cada arquivo

In [53]:
def parse_xml(file_path):
    """
    Parsea um arquivo XML e retorna as anotações relevantes.
    Funciona para ambos os tipos de XML fornecidos.
    """
    tree = ET.parse(file_path)
    root = tree.getroot()

    # Recuperar o nome do arquivo de áudio
    audio_file_name = root.find(".//audioFileName").text

    # Recuperar os eventos de transcrição
    annotations = []
    for event in root.findall(".//event"):
        onset = float(event.find("onsetSec").text)
        offset = float(event.find("offsetSec").text)
        pitch = int(event.find("pitch").text)
        fret_number = int(event.find("fretNumber").text)
        string_number = int(event.find("stringNumber").text)
        excitation_style = event.find("excitationStyle").text
        expression_style = event.find("expressionStyle").text

        annotations.append({
            "onset": onset,
            "offset": offset,
            "pitch": pitch,
            "fret_number": fret_number,
            "string_number": string_number,
            "excitation_style": excitation_style,
            "expression_style": expression_style,
        })

    return audio_file_name, annotations

## Uniformização de dados

Transforma todos os dados em um mesmo comprimento (duração)

In [54]:
def pad_or_truncate_spectrogram(spectrogram, max_len=SPEC_MAX_LEN):
    """
    Ajusta o tamanho do espectrograma para um comprimento fixo.
    - Trunca se for maior.
    - Adiciona zero-padding se for menor.
    """
    if spectrogram.shape[1] > max_len:
        return spectrogram[:, :max_len]
    else:
        pad_width = max_len - spectrogram.shape[1]
        return np.pad(spectrogram, ((0, 0), (0, pad_width)), mode="constant")

## Processamento do audio

In [55]:
def process_audio(file_path, annotations):
    """
    Carrega o áudio e extrai espectrogramas para cada evento anotado.
    """
    audio, sr = librosa.load(file_path, sr=SAMPLE_RATE, mono=True)

    processed_data = []
    for annotation in annotations:
        onset_sample = int(annotation["onset"] * sr)
        offset_sample = int(annotation["offset"] * sr)

        # Extrai o trecho de áudio correspondente ao evento
        clip = audio[onset_sample:offset_sample]

        # Normaliza o áudio
        clip = librosa.util.normalize(clip)

        # Calcula o espectrograma Mel
        spectrogram = librosa.feature.melspectrogram(
            y=clip, sr=SAMPLE_RATE, n_fft=N_FFT, hop_length=HOP_LENGTH
        )
        spectrogram_db = librosa.power_to_db(spectrogram, ref=np.max)

        # Normalização e ajuste de comprimento
        spectrogram_db = normalize_spectrogram(spectrogram_db)
        spectrogram_db = pad_or_truncate_spectrogram(spectrogram_db)

        processed_data.append({
            "spectrogram": spectrogram_db,
            "metadata": annotation
        })

    return processed_data

In [56]:
def save_data(processed_data, output_path, split):
    """
    Salva os espectrogramas e metadados em formato estruturado.
    """
    spectrograms = []
    metadata = []

    for data in processed_data:
        spectrograms.append(data["spectrogram"])
        metadata.append(data["metadata"])

    # Salva os espectrogramas em formato comprimido
    np.savez_compressed(os.path.join(output_path, f"spectrograms_{split}.npz"),
                        spectrograms=spectrograms)

    # Salva os metadados em formato JSON
    with open(os.path.join(output_path, f"metadata_{split}.json"), "w") as f:
        json.dump(metadata, f)

In [57]:
def preprocess_and_leave_one_out(dataset_path, output_path, leave_out_category):
    """
    Processa o dataset completo usando Leave-One-Out por Categoria.
    """
    os.makedirs(output_path, exist_ok=True)

    all_processed_data = []

    for category in os.listdir(dataset_path):
        category_path = os.path.join(dataset_path, category)

        if os.path.isdir(category_path):
            print(f"Processando categoria: {category}")

            annotation_path = os.path.join(category_path, "annotation")
            audio_path = os.path.join(category_path, "audio")

            if not os.path.exists(annotation_path) or not os.path.exists(audio_path):
                print(f"Pasta 'annotation' ou 'audio' não encontrada em {category_path}")
                continue

            for annotation_file in os.listdir(annotation_path):
                if annotation_file.endswith(".xml"):
                    annotation_full_path = os.path.join(annotation_path, annotation_file)

                    # Parse do XML para obter as anotações
                    audio_file_name, annotations = parse_xml(annotation_full_path)
                    audio_full_path = os.path.join(audio_path, audio_file_name)

                    if not os.path.exists(audio_full_path):
                        print(f"Arquivo de áudio não encontrado: {audio_full_path}")
                        continue

                    # Processa o áudio
                    processed_data = process_audio(audio_full_path, annotations)
                    
                    # Adiciona a categoria em cada anotação
                    for data in processed_data:
                        data["metadata"]["category"] = category
                        
                    all_processed_data.extend(processed_data)

    # Filtra os dados para Leave-One-Out
    train_data = [data for data in all_processed_data if data["metadata"]["category"] != leave_out_category]
    test_data = [data for data in all_processed_data if data["metadata"]["category"] == leave_out_category]

    # Divide os dados de treino em treino e validação
    train_data, val_data = train_test_split(train_data, test_size=0.2, random_state=42)

    # Salva os conjuntos
    save_data(train_data, output_path, "train")
    save_data(val_data, output_path, "val")
    save_data(test_data, output_path, "test")

In [58]:
# Executa o pré-processamento com Leave-One-Out
preprocess_and_leave_one_out(DATASET_PATH, OUTPUT_PATH, LEAVE_OUT_CATEGORY)

Processando categoria: Fender Strat Clean Neck SC
Processando categoria: Fender Strat Clean Neck SC Chords
Processando categoria: Ibanez Power Strat Clean Bridge HU
Processando categoria: Ibanez Power Strat Clean Bridge HU Chords
Processando categoria: Ibanez Power Strat Clean Bridge+Neck SC
Processando categoria: Ibanez Power Strat Clean Neck HU
Processando categoria: preprocessed_data
Pasta 'annotation' ou 'audio' não encontrada em ../data/IDMT-SMT-GUITAR_V2/dataset1\preprocessed_data
